In [5]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.integrate as INT



In [31]:
def repetitive_stim_maker(num_repeat,total_time,off_first = False):
    '''
    num_repeat: number of segments each made of repetitive cycles (e.g., if 2, [11110000])\\
    total_time: span of the stimulation experiment. For example, if 20, it means 20 cycles (and for a 5 min stim sampling, 100 mins).\\
    off_first: bool whether stim starts with off signal or not. Defaults is False.\\
    **Example**:\\
    num_repeat = 4 and total_time = 60 means [[15 ones and 15 zeros, 15 ones and 15 zeros]
    '''
    num_reps = int(np.floor(total_time/num_repeat))
    ons = np.repeat(1,num_reps)
    offs = np.repeat(0,num_reps)
    tile = np.hstack((offs,ons)) if off_first else np.hstack((ons,offs)) 
    stim_vec = np.tile(tile,int(num_repeat))
    stim_vec = stim_vec[:total_time]
    return stim_vec


In [176]:
def rxn1(x,t,params,stim_vec):
    alpha , k , n , tau_delay , h1 , h2 ,c2 , delta = params
    H,E,F = x
    U = stim_vec[int(t/5)]
    L = ((c2*H)**n)/(k + (c2*H)**n)
    return [
        U- c2*H,
        h1 - h2*E,
        alpha * E * L - delta * F
    ]
    
def rxn2(x,t,params,params_tet,stim_vec):
    alpha , k , n , tau_delay , h1 , h2 ,c2 , delta_T = params
    alpha , beta, k_tet , k , n ,n_tet, tau_delay , h1 , h2 , c2,delta = params_tet
    H,E,T,F = x
    U = stim_vec[max(int((t-tau_delay)/5),0)]
    L = ((c2*H)**n)/(k + (c2*H)**n)
    return [
        U- c2*H,
        h1 - h2*E,
        alpha * E * L - delta_T * T,
        beta*E/(1+((T)/k_tet)**n_tet) - delta * F
    ]
    
def rxn3(x,t,params,params_tet,stim_vec):
    alpha , k , n , tau_delay , h1 , h2 ,c2 , delta_T = params
    alpha , beta, k_tet , k , n ,n_tet, tau_delay , h1 , h2 , c2,delta,s_r,s_tot = params_tet
    H,E,T,F,S = x
    U = stim_vec[max(int((t-tau_delay)/5),0)]
    L = ((c2*H)**n)/(k + (c2*H)**n)
    return [
        1*U- c2*H,
        h1 - h2*E,
        alpha * E * L - delta_T * T - S*T*s_r,
        beta*E/(1+((T)/k_tet)**n_tet) - delta * F,
        -S*T*s_r + delta_T*(s_tot-S)
    ]

In [ ]:
delta = 0.01
alpha = 1.75
beta = 1
k = 0.4851
k_tet = 100
h1 = 2.3435*0.0303
h2 = 0.0303
tau_delay = 12
n=3.6
n_tet = 2
c2 = 0.0631
s_r = 10
s_tot = 35
params = [alpha , k , n , tau_delay , h1 , h2 ,c2 , delta]
params_tet = [alpha , beta, k_tet , k , n ,n_tet, tau_delay , h1 , h2 , c2,delta,s_r,s_tot]


t = np.linspace(0,1000,10000)
x0 = [0,round(np.random.poisson(h1/h2)),0,0,s_tot]
stim_vec = repetitive_stim_maker(10,int(1000/5)+2)
sol = INT.odeint(rxn3,x0,t,args = (params,params_tet,stim_vec))

fig, ax = plt.subplots(figsize=(8, 4))
for i, val in enumerate(stim_vec):
        x_start =  i * 50
        x_end = x_start + 50
        color = 'green' if val == 1 else 'red'
        ax.axvspan(x_start, x_end, facecolor=color, alpha=0.2)

plt.plot(sol[:,2])
plt.plot(sol[:,3])
plt.show()

In [ ]:
class Simple_spring_mass():
    def __init__(self,m,k,c,xr = 0,x0=0,v0=0,g=10,dt=0.1):
        self.m = m
        self. k = k
        self.c = c
        self.xr= xr
        self.v0 = v0
        self.x0 = x0
        self.g = g
        self.dt = dt
        
        self.x = [x0]
        self.v = [v0]
        self.U = [0]
        self.time = 0
        return
    
    def __str__(self):
        sampling = int(1/self.dt)
        if self.time < 5:
            i = int(np.floor(self.time))
            past_forces = np.array(self.U)[-1*np.arange(1,i*sampling,sampling)][::-1]
            past_positions = np.array(self.x)[-1*np.arange(1,i*sampling,sampling)][::-1]
            past_velocities = np.array(self.v)[-1*np.arange(1,i*sampling,sampling)][::-1]
        else: 
            i = 5
            past_forces = np.array(self.U)[-1*np.arange(1,i*sampling,sampling)][::-1]
            past_positions = np.array(self.x)[-1*np.arange(1,i*sampling,sampling)][::-1]
            past_velocities = np.array(self.v)[-1*np.arange(1,i*sampling,sampling)][::-1]
        return f'''
                        System description:
                            mass (m)= {self.m}
                            spring constant (k)= {self.k}
                            damper constant (c)= {self.c}
                            spring resting location (x_r)= {self.xr}
                            gravity constant (g) = {self.g}
                            current position (x) = {self.x[-1]}
                            current velocity (v) = {self.v[-1]}
                            current force (u) = {self.U[-1]}
                            current time (t) = {self.time} s
                            forces applied in the past {i} s = {past_forces}
                            position in the past {i} s = {past_positions}
                            velocity applied in the past {i} s = {past_velocities}
                        System dynamics: 
                            dv = dt/m *(-k*(x-x_r) - v*c + u + m*g) 
                            dx = dt*v
                            v[t+1] = v[t] + dv
                            x[t+1] = x[t] + dx
        '''
    def exert(self,u):
        self.U.append(u)
        dv = self.dt/self.m*(-self.k*(self.x[-1]-self.xr)-self.v[-1]*self.c+u+self.m*self.g)
        v_t = self.v[-1] + dv
        dx = v_t*self.dt
        x_t = self.x[-1] + dx
        self.x.append(x_t)
        self.v.append(v_t)
        self.time+= self.dt
    
    def control(self,x_g,p):
        #closed loop (proportional-integral)
        if self.x[-1]== x_g : return
        else: 
            hist = self.x
            errors = [x_g - x for x in hist]
            integral = np.sum(errors)*self.dt
            self.exert(p*integral)
            return p*integral
    
    def state(self):
        return self.x,self.v
        
model = Simple_spring_mass(1,5,1,x0=-2,g=10)
print(model)
U = []
x_g = 5
t = 1
while model.time <20:
    u = np.sin(t)
    while model.time <t:
        model.exert(u)
    t+=1
    print(model)
    
x,v = model.state()

plt.plot(model.U)
plt.plot(x)
plt.plot(v)
        

        
        

        